# Qwen3 0.6B Unmodified Model Baseline

Ten notebook uruchamia ewaluację modelu bazowego `Qwen/Qwen3-0.6B` bez adapterów LoRA. Pomiar jest przygotowany tak, aby odpowiadał ewaluacji punktów kontrolnych metody `03_causal_lm_next_token`: te same zbiory, limit próbek, seed oraz zestaw metryk. Domyślnie używany jest wariant promptu `defined-labels`, który doprecyzowuje znaczenie klas dla modelu bez dostrajania.

Sprawdzane są dwa warianty interpretacji odpowiedzi modelu bazowego:

- generowanie odpowiedzi i możliwie łagodne parsowanie etykiety,
- ocena prawdopodobieństwa następnego tokenu `ham` oraz `spam`.


In [ ]:
from pathlib import Path
import subprocess
import sys


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "dataset").is_dir() and (candidate / "lora-fine-tuning").is_dir():
            return candidate
    raise RuntimeError("Could not find project root.")


PROJECT_ROOT = find_project_root()
METHOD_DIR = PROJECT_ROOT / "lora-fine-tuning" / "methods" / "00_unmodified_model"
SCRIPT = METHOD_DIR / "qwen3_0.6b_unmodified_baseline.py"
RESULTS_ROOT = METHOD_DIR / "results"

RUN_ID = None
DATASETS = ["train_subset", "enron", "fraudulent_email_corpus", "spam_ham"]
SAMPLE_LIMIT = 2000
MODE = "both"  # generation, next-token, or both
BATCH_SIZE = 32
MAX_SEQ_LENGTH = 1024
MAX_NEW_TOKENS = 24
PROMPT_STYLE = "defined-labels"
SEED = 67

print(f"Project root: {PROJECT_ROOT}")
print(f"Script:       {SCRIPT}")
print(f"Results:      {RESULTS_ROOT}")


## Uruchomienie ewaluacji

Pełny pomiar zapisuje `summary.csv`, `metrics.json` oraz predykcje dla każdej pary metoda-zbiór. Predykcje są szczególnie ważne dla wariantu generacyjnego, ponieważ pozwalają sprawdzić, jakie odpowiedzi model bazowy zwraca poza oczekiwanym szablonem.


In [ ]:
command = [
    sys.executable,
    str(SCRIPT),
    "--results-root", str(RESULTS_ROOT),
    "--datasets", *DATASETS,
    "--sample-limit", str(SAMPLE_LIMIT),
    "--mode", MODE,
    "--batch-size", str(BATCH_SIZE),
    "--max-seq-length", str(MAX_SEQ_LENGTH),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
    "--prompt-style", PROMPT_STYLE,
    "--seed", str(SEED),
    "--write-predictions",
]
if RUN_ID:
    command.extend(["--run-id", RUN_ID])

print(" ".join(command))
subprocess.run(command, cwd=PROJECT_ROOT, check=True)


## Podsumowanie metryk

Komórka automatycznie znajduje najnowszy run, wczytuje `summary.csv` i pokazuje te same metryki, które są wykorzystywane przy analizie punktów kontrolnych.


In [ ]:
import pandas as pd

runs = sorted((RESULTS_ROOT / "runs").glob("qwen3_0p6b_unmodified_*"), key=lambda p: p.stat().st_mtime)
if not runs:
    raise FileNotFoundError("No baseline runs found.")
RUN_DIR = runs[-1]
summary_path = RUN_DIR / "summary.csv"
summary = pd.read_csv(summary_path)
print(RUN_DIR)

columns = [
    "dataset", "method", "rows", "ham_count", "spam_count",
    "accuracy", "precision", "recall", "f1", "specificity",
    "balanced_accuracy", "false_positive_rate", "false_negative_rate",
    "spam_prediction_rate", "parse_failure_rate",
]
display(summary[columns].sort_values(["method", "dataset"]))


## Inspekcja odpowiedzi generacyjnych

Parser baseline'u celowo daje modelowi możliwie dużą szansę: rozpoznaje JSON, pola typu `label: spam`, samą etykietę, odpowiedzi opisowe typu `this is spam`, ale również zaprzeczenia takie jak `not spam`. Poniższy podgląd pomaga ocenić, które reguły były wykorzystywane najczęściej i gdzie parser nadal nie potrafił znaleźć etykiety.


In [ ]:
generation_files = sorted((RUN_DIR / "predictions").glob("*_generation_parsing.csv"))
if not generation_files:
    raise FileNotFoundError("No generation prediction files found.")

generation = pd.concat((pd.read_csv(path) for path in generation_files), ignore_index=True)
print("Parse rules")
display(generation.groupby(["dataset", "parse_rule"]).size().rename("count").reset_index())

preview_columns = [
    "dataset", "label_text", "prediction_text", "correct", "parse_failed",
    "parse_rule", "subject", "raw_generation",
]
display(generation.loc[generation["parse_failed"] | ~generation["correct"], preview_columns].head(30))


## Tabela do pracy

Ta komórka przygotowuje krótką tabelę LaTeX z wynikami modelu bazowego. Po pełnym uruchomieniu można ją wykorzystać w rozdziale jako punkt odniesienia dla metod dostrajanych.


In [ ]:
TABLE_PATH = PROJECT_ROOT / "thesis" / "tex" / "tables" / "unmodified_baseline_metrics.tex"
metric_view = summary[["dataset", "method", "accuracy", "f1", "recall", "specificity", "balanced_accuracy", "spam_prediction_rate"]].copy()
for column in ["accuracy", "f1", "recall", "specificity", "balanced_accuracy", "spam_prediction_rate"]:
    metric_view[column] = (100 * metric_view[column]).round(1)

label_map = {
    "train_subset": r"\texttt{train\_subset}",
    "enron": r"\texttt{enron}",
    "fraudulent_email_corpus": r"\texttt{fraudulent}",
    "spam_ham": r"\texttt{spam\_ham}",
    "generation_parsing": "generowanie",
    "next_token": "następny token",
}

lines = [
    r"\begin{tabular}{llrrrrrr}",
    r"\hline",
    "Zbiór & Wariant & Accuracy & F1 & Recall & Specificity & Balanced acc. & Spam pred. \\\\",
    r"\hline",
]
for _, row in metric_view.sort_values(["method", "dataset"]).iterrows():
    lines.append(
        f"{label_map[row['dataset']]} & {label_map[row['method']]} & "
        f"{row['accuracy']:.1f} & {row['f1']:.1f} & {row['recall']:.1f} & "
        f"{row['specificity']:.1f} & {row['balanced_accuracy']:.1f} & {row['spam_prediction_rate']:.1f} \\\\"
    )
lines.extend([r"\hline", r"\end{tabular}"])
TABLE_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(TABLE_PATH)
